James’s EDA found make % drops in the last 2–10 seconds of a quarter (heaves), the 4th quarter is a bit worse, and home teams shoot about 0.7 points better.

Flags: last 2 seconds, last 10 seconds, 4th quarter, overtime, home.
Data integration: add the score margin before each shot from play-by-play (the nbastats files from the shufinskiy repo), matched by GAME_EVENT_ID once Ruifeng keeps it. That repo stops at 2024–25, so 2025–26 play-by-play will have to come from Ruifeng’s NBA API access.
Test: does score margin help on top of the time flags?


# Feature Extraction

In [1]:
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd

df = pd.read_csv("datasets/train_with_player_bio.csv")
# change data points with 12 minutes left to 11 minutes left
df["MINS_LEFT"] = df["MINS_LEFT"].replace(12, 11)

print(df.head())

   SEASON_1 SEASON_2     TEAM_ID      TEAM_NAME  PLAYER_ID        PLAYER_NAME  \
0      2022  2021-22  1610612737  Atlanta Hawks    1628381       John Collins   
1      2022  2021-22  1610612737  Atlanta Hawks    1629631    De'Andre Hunter   
2      2022  2021-22  1610612737  Atlanta Hawks    1629027         Trae Young   
3      2022  2021-22  1610612737  Atlanta Hawks    1628381       John Collins   
4      2022  2021-22  1610612737  Atlanta Hawks     203992  Bogdan Bogdanović   

  POSITION_GROUP POSITION   GAME_DATE   GAME_ID  ... IS_THREE PLAYER_HEIGHT  \
0              F      F-C  10-21-2021  22100014  ...        0           6-9   
1              F      F-G  10-21-2021  22100014  ...        0           6-8   
2              G        G  10-21-2021  22100014  ...        0           6-1   
3              F      F-C  10-21-2021  22100014  ...        0           6-9   
4              G        G  10-21-2021  22100014  ...        1           6-6   

  PLAYER_HEIGHT_INCHES  PLAYER_WEIGHT 

In [2]:
# create new feature that flags if a shot is taken in the last two seconds
df["LAST_2_SECONDS"] = np.where(df["SECS_LEFT"] <= 2, 1, 0)

# create new feature that flags if a shot is taken in the last ten seconds
df["LAST_10_SECONDS"] = np.where(df["SECS_LEFT"] <= 10, 1, 0)


Since every shot in the last two seconds is also a shot in the last ten seconds, these features are highly correlated which may cause some issues with some models.

In [3]:
# flags when it is the fourth quarter
df["FOURTH_QUARTER"] = np.where(df["QUARTER"] == 4, 1, 0)

# create team abbreviation feature
team_abbreviations = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "LA Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

df["TEAM_ABV"] = df["TEAM_NAME"].map(team_abbreviations)

# create new feature for home team
df["IS_HOME"] = np.where(df["TEAM_ABV"] == df["HOME_TEAM"], 1, 0) #if the team name matches the home team, set IS_HOME to 1, otherwise 0

This creates some new features used for the model: last 2 seconds, last 10 seconds, fourth quarter, and if its at home.

# Logistic Regression

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

train_df = df[df["SEASON"].isin(["2021-22", "2022-23", "2023-24"])]
test_df = df[df["SEASON"] == "2024-25"]

features = [
    "SHOT_DISTANCE",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

Ruifeng’s baselines trained on 2021–22 to 2023–24 and checked their scores on 2024–25:

Model | Accuracy on 2024–25 | AUC

Always guess “miss” | 53.3% | 0.50

Distance only | 60.3% | 0.642

Version 1 (7 number features) | 60.8% | 0.641

Adding 6 more features barely helped. That’s because Version 1 used only number columns. 


- Same split as Ruifeng’s baselines: train on 2021–22 to 2023–24, check results on 2024–25.
- Nobody touches 2025–26.
- Same 4 scores for every experiment: accuracy, AUC, log loss and Brier score. The last two measure the class probabilities our professor asked for.
- Anything that learns from the data (averages, player shooting %, scaling) is calculated from training seasons only.
- Each person adds one row per experiment to a shared results table.


In [5]:
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Baseline Logistic Regression with Shot Distance")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Baseline Logistic Regression with Shot Distance
Accuracy: 0.603
AUC: 0.6419
Log Loss: 0.6632
Brier Score: 0.2354


In [6]:
features = [
    "SHOT_DISTANCE",
    "QUARTER",
    "SECS_LEFT",
    "LAST_2_SECONDS",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Quarter, Secs Left, and Last 2 Seconds")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Quarter, Secs Left, and Last 2 Seconds
Accuracy: 0.6022
AUC: 0.6415
Log Loss: 0.6631
Brier Score: 0.2353


In [7]:
features = [
    "SHOT_DISTANCE",
    "QUARTER",
    "SECS_LEFT",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Quarter, Secs Left, Last 2 Seconds, and Last 10 Seconds")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Quarter, Secs Left, Last 2 Seconds, and Last 10 Seconds
Accuracy: 0.6023
AUC: 0.6414
Log Loss: 0.6631
Brier Score: 0.2353


In [8]:
features = [
    "SHOT_DISTANCE",
    "QUARTER",
    "SECS_LEFT",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Quarter, Secs Left, Last 2 Seconds, Last 10 Seconds, and Fourth Quarter")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Quarter, Secs Left, Last 2 Seconds, Last 10 Seconds, and Fourth Quarter
Accuracy: 0.602
AUC: 0.6413
Log Loss: 0.6631
Brier Score: 0.2353


In [9]:
features = [
    "SHOT_DISTANCE",
    "QUARTER",
    "SECS_LEFT",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "IS_HOME",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Quarter, Secs Left, Last 2 Seconds, Last 10 Seconds, Fourth Quarter, and Is Home")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Quarter, Secs Left, Last 2 Seconds, Last 10 Seconds, Fourth Quarter, and Is Home
Accuracy: 0.6024
AUC: 0.6413
Log Loss: 0.6631
Brier Score: 0.2353


These features don't really improve the model by much

## Baseline Logistic Regression with IS_THREE

In [10]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three
Accuracy: 0.6071
AUC: 0.6401
Log Loss: 0.6609
Brier Score: 0.2343


Adding the IS_THREE feature slightly improves the accuracy of the logistic regression model.

In [11]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT"
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left
Accuracy: 0.6071
AUC: 0.6403
Log Loss: 0.6609
Brier Score: 0.2343


In [12]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home
Accuracy: 0.607
AUC: 0.6403
Log Loss: 0.6609
Brier Score: 0.2343


In [13]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds
Accuracy: 0.6072
AUC: 0.6403
Log Loss: 0.6609
Brier Score: 0.2343


In [14]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER"
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter
Accuracy: 0.6073
AUC: 0.6403
Log Loss: 0.6609
Brier Score: 0.2342


## Scaling Numeric Features

In [15]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

preprocessor = make_column_transformer(
    (StandardScaler(), ["SHOT_DISTANCE", "QUARTER", "SECS_LEFT"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER"
]


x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter
Accuracy: 0.6073
AUC: 0.6403
Log Loss: 0.6608
Brier Score: 0.2342


It seems like scaling the numerical features has a minimal effect on improving the model.

In [16]:
preprocessor = make_column_transformer(
    (StandardScaler(), ["SHOT_DISTANCE", "SECS_LEFT"]),
    (OneHotEncoder(handle_unknown="ignore"), ["HOME_TEAM", "QUARTER"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "HOME_TEAM",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter, Home Team")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))


Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter
Accuracy: 0.6071
AUC: 0.6408
Log Loss: 0.6607
Brier Score: 0.2342


One hot encoding the HOME_TEAM feature allows the model to take into account which arena the shot takes place in because some arenas, like Denver, affect fg%. Minimal improvement of the model.

# Decision Tree

In [17]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5, random_state=42)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance and Is Three (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance and Is Three (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance and Is Three (Train)
Accuracy: 0.6247
AUC: 0.6435
Log Loss: 0.6514
Brier Score: 0.2298 

Decision Trees with Shot Distance and Is Three (Test)
Accuracy: 0.624
AUC: 0.6418
Log Loss: 0.6521
Brier Score: 0.2302


Decision Trees perform better than logistic regression with only shot distance and is three. I experimented with different max depths of the tree and it seems like 5-8 is a good depth range balance between a weak decision tree and overfitting.

In [18]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT"
]

model = DecisionTreeClassifier(max_depth=5, random_state=42)

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left (Train)
Accuracy: 0.6252
AUC: 0.6437
Log Loss: 0.6513
Brier Score: 0.2298 

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left (Test)
Accuracy: 0.6227
AUC: 0.6413
Log Loss: 0.6522
Brier Score: 0.2302


Quarter and Seconds Left makes the model slightly worse

In [19]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
]

model = DecisionTreeClassifier(max_depth=5, random_state=42)

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Train)
Accuracy: 0.6252
AUC: 0.6437
Log Loss: 0.6513
Brier Score: 0.2298 

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Test)
Accuracy: 0.6227
AUC: 0.6413
Log Loss: 0.6522
Brier Score: 0.2302


In [20]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
]

model = DecisionTreeClassifier(max_depth=5, random_state=42)

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Train)
Accuracy: 0.6252
AUC: 0.6437
Log Loss: 0.6513
Brier Score: 0.2298 

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home (Test)
Accuracy: 0.6227
AUC: 0.6413
Log Loss: 0.6522
Brier Score: 0.2302


In [21]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
]

model = DecisionTreeClassifier(max_depth=5, random_state=42)

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds (Train)
Accuracy: 0.6252
AUC: 0.6437
Log Loss: 0.6513
Brier Score: 0.2298 

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds (Test)
Accuracy: 0.6227
AUC: 0.6413
Log Loss: 0.6522
Brier Score: 0.2302


In [22]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER"
]

model = DecisionTreeClassifier(max_depth=5, random_state=42)

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter (Train)
Accuracy: 0.6252
AUC: 0.6437
Log Loss: 0.6513
Brier Score: 0.2298 

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter (Test)
Accuracy: 0.6227
AUC: 0.6413
Log Loss: 0.6522
Brier Score: 0.2302


## One Hot Encoding Home Team

In [23]:
preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["HOME_TEAM"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "HOME_TEAM",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter, Home Team (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter, Home Team (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter, Home Team (Train)
Accuracy: 0.6276
AUC: 0.6446
Log Loss: 0.6502
Brier Score: 0.2293 

Decision Trees with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 2/10 Seconds, Fourth Quarter, Home Team (Test)
Accuracy: 0.6239
AUC: 0.6418
Log Loss: 0.6516
Brier Score: 0.2299


One hot encoding Home Team so that each arena is factored into fg% helps improve the model and is probably the most predictive feature out of all the other game situation stats

In [24]:
preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["HOME_TEAM"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "IS_HOME",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "HOME_TEAM",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance, Is Three, Is Home, Last 10 Seconds, Fourth Quarter, Home Team (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance, Is Three, Is Home, Last 10 Seconds, Fourth Quarter, Home Team (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance, Is Three, Is Home, Last 10 Seconds, Fourth Quarter, Home Team (Train)
Accuracy: 0.6276
AUC: 0.6445
Log Loss: 0.6502
Brier Score: 0.2293 

Decision Trees with Shot Distance, Is Three, Is Home, Last 10 Seconds, Fourth Quarter, Home Team (Test)
Accuracy: 0.624
AUC: 0.6424
Log Loss: 0.6514
Brier Score: 0.2299


This is the most accurate model that I could make (with a max depth of 5 and no other hyperparameters), removing quarter and seconds left actually improved the model. Last 2 seconds and last 10 seconds are mostly redundant and removing the last 2 seconds feature marginally improved the model.

In [25]:
preprocessor = make_column_transformer(
    (StandardScaler(), ["SHOT_DISTANCE"]),
    (OneHotEncoder(handle_unknown="ignore"), ["HOME_TEAM"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "HOME_TEAM",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter, Home Team")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Shot Distance, Is Three, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter
Accuracy: 0.607
AUC: 0.6408
Log Loss: 0.6607
Brier Score: 0.2342


Removing Quarter and Seconds Left on logistic regression did not improve the model unlike with decision trees.

# Hyperparameter Tuning

for later, why does removing quarter and seconds affect the decision tree and not logistic regression

## Logistic Regression

In [53]:
features = [
    "SHOT_DISTANCE",
    "IS_THREE",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model = LogisticRegression(max_iter=1000, C=100, solver="newton-cholesky")
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Quarter, Secs Left, and Last 2 Seconds")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Logistic Regression with Quarter, Secs Left, and Last 2 Seconds
Accuracy: 0.6071
AUC: 0.6401
Log Loss: 0.6609
Brier Score: 0.2343


Modifying the solver and C values don't improve the model at all (with these features)

In [82]:
preprocessor = make_column_transformer(
    (StandardScaler(), ["SHOT_DISTANCE", "SECS_LEFT"]),
    (OneHotEncoder(handle_unknown="ignore"), ["HOME_TEAM", "QUARTER"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000, C=0.01, l1_ratio=1, solver="saga")
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "QUARTER",
    "SECS_LEFT",
    "IS_HOME",
    "LAST_2_SECONDS",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "HOME_TEAM",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter, Home Team")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))


Logistic Regression with Shot Distance, Is Three, Quarter, Secs Left, Is Home, Last 10 Seconds, Last 2 Seconds, Fourth Quarter, Home Team
Accuracy: 0.607
AUC: 0.641
Log Loss: 0.6607
Brier Score: 0.2342


A C value of 0.01 slightly improves the AUC while the log loss and brier score doesn't move. Changing the L1 ratio, solver, and increasing max iterations don't improve the model.

## Decision Trees

In [102]:
model = DecisionTreeClassifier(max_depth=5, random_state=42, min_samples_split=2, min_samples_leaf=1)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
]

x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance and Is Three (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance and Is Three (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance and Is Three (Train)
Accuracy: 0.6247
AUC: 0.6435
Log Loss: 0.6514
Brier Score: 0.2298 

Decision Trees with Shot Distance and Is Three (Test)
Accuracy: 0.624
AUC: 0.6418
Log Loss: 0.6521
Brier Score: 0.2302


- min samples split doesn't affect the accuracy of the model, and increasing max depth with it causes overfitting
- adjusting min samples leaft doesn't change the model
- entropy and log loss splitting criterion makes the model worse

In [122]:
preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["HOME_TEAM"]),
    remainder="passthrough"
)

model = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(max_depth=5, random_state=42, max_features=5)
)

features = [
    "SHOT_DISTANCE",
    "IS_THREE",
    "IS_HOME",
    "LAST_10_SECONDS",
    "FOURTH_QUARTER",
    "HOME_TEAM",
]
x_train = train_df[features]
y_train = train_df["MADE"] # target

x_test = test_df[features]
y_test = test_df["MADE"]

model.fit(x_train, y_train)

# training predictions
train_pred = model.predict(x_train)
train_prob = model.predict_proba(x_train)[:, 1]

y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:, 1]

# evaluate
train_accuracy = accuracy_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_prob)
train_loss = log_loss(y_train, train_prob)
train_brier = brier_score_loss(y_train, train_prob)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
loss = log_loss(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Decision Trees with Shot Distance and Is Three (Train)")
print("Accuracy:", round(train_accuracy, 4))
print("AUC:", round(train_auc, 4))
print("Log Loss:", round(train_loss, 4))
print("Brier Score:", round(train_brier, 4), "\n")

print("Decision Trees with Shot Distance and Is Three (Test)")
print("Accuracy:", round(accuracy, 4))
print("AUC:", round(auc, 4))
print("Log Loss:", round(loss, 4))
print("Brier Score:", round(brier, 4))

Decision Trees with Shot Distance and Is Three (Train)
Accuracy: 0.5871
AUC: 0.603
Log Loss: 0.6698
Brier Score: 0.2386 

Decision Trees with Shot Distance and Is Three (Test)
Accuracy: 0.5864
AUC: 0.5992
Log Loss: 0.6705
Brier Score: 0.2389


- increasing max depth to 8 (from 5) improved the accuracy, AUC, and Brier Score, but leads to more overfitting
- min samples leaf, min sample split doesn't change anything
- splitting criterion makes it slightly worse
- sqrt, log2 max features heavily decreases the predictability of the model
- decreasing max features makes the model worse in general